In [ ]:
import time
import torch
import torch.nn as nn
import sys
import os

# 1. 프로젝트 루트 경로 설정
project_root = "/home/jgryu/workspace/weight_compression/quip-sharp"
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root added to path: {project_root}")

# 2. 필요한 모듈 임포트
from latticee8_padded12_rvq4bit import E8P12RVQ4B_codebook, QuantizedE8P12RVQ4BLinear, matmul_hadU_cuda
from lib.utils import get_hadK

def fast_quantize_for_bench(layer, codebook, W):
    """
    디코딩 속도 측정을 위해 복잡한 QUIP 알고리즘(Hessian, LDLQ 등)을 건너뛰고
    단순히 가중치를 양자화하여 Qidxs_list를 생성합니다.
    """
    print("Fast quantizing for benchmark (skipping Hessian/LDLQ)...")
    
    orig_shape = W.shape
    flattened_W = W.view(-1, codebook.codesz)
    
    with torch.no_grad():
        _, idxs = codebook.quantize(flattened_W, return_idx=True)
    
    idxs = idxs.view(orig_shape[0], orig_shape[1] // codebook.codesz)
    packed_idxs = codebook.maybe_pack_idxs(idxs)
    Qidxs_list = layer.maybe_unpack_idxs(packed_idxs)
    
    return Qidxs_list

def benchmark_e8p_cache_wh_with_stats():
    # 1. 설정 (Setup)
    # 코드에 적힌대로 cuda:0 사용 (환경에 따라 수정 가능)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if device.type != 'cuda':
        print("Warning: CUDA is not available. Benchmark results will be invalid.")
        return

    n = 4096
    m = 4096
    
    print(f"Initializing Benchmark for {n}x{m} Tensor...")

    # 2. 모델 및 코드북 초기화
    quantizer_codebook = E8P12RVQ4B_codebook(inference=False).to(device)
    linear_layer = QuantizedE8P12RVQ4BLinear(device).to(device)

    # 3. 더미 데이터 생성 및 양자화
    print("Quantizing tensor...")
    W = torch.randn(n, m, device=device, dtype=torch.float32)
    Qidxs_list = fast_quantize_for_bench(linear_layer, quantizer_codebook, W)

    # 4. QuantizedLinear 로직을 참고하여 K 및 Hadamard 행렬 결정
    print("Determining K and Hadamard matrices using get_hadK...")
    
    had_left_raw, K_left = get_hadK(n)
    had_right_raw, K_right = get_hadK(m)
    
    had_left = had_left_raw.to(device).contiguous() if had_left_raw is not None else None
    had_right = had_right_raw.to(device).contiguous() if had_right_raw is not None else None
    
    print(f"K_left: {K_left}, K_right: {K_right}")

    # 5. 벤치마크 (Benchmark cache_WH)
    print(f"Benchmarking cache_WH on {device}...")
    
    # Warmup
    for _ in range(100):
        linear_layer.cache_WH(
            n=n, m=m, 
            Qidxs_list=Qidxs_list, 
            had_left=had_left, 
            had_right=had_right, 
            K_left=K_left, 
            K_right=K_right
        )
    torch.cuda.synchronize()

    # 측정 루프 설정
    num_iters = 100
    
    # 개별 측정을 위한 이벤트 리스트 생성
    start_events = [torch.cuda.Event(enable_timing=True) for _ in range(num_iters)]
    end_events = [torch.cuda.Event(enable_timing=True) for _ in range(num_iters)]

    # 실제 실행 및 기록
    for i in range(num_iters):
        start_events[i].record()
        
        linear_layer.cache_WH(
            n=n, m=m, 
            Qidxs_list=Qidxs_list, 
            had_left=had_left, 
            had_right=had_right, 
            K_left=K_left, 
            K_right=K_right
        )
        
        end_events[i].record()
    
    # 모든 커널이 끝날 때까지 대기
    torch.cuda.synchronize()

    # 6. 통계 계산 (평균 및 표준편차)
    times_ms = [s.elapsed_time(e) for s, e in zip(start_events, end_events)]
    times_tensor = torch.tensor(times_ms)
    
    avg_time_ms = times_tensor.mean().item()
    std_dev_ms = times_tensor.std().item()

    print(f"--------------------------------------------------")
    print(f"Task: cache_WH Execution")
    print(f"Matrix Size: {n}x{m}")
    print(f"K Parameters: Left={K_left}, Right={K_right}")
    print(f"Iterations: {num_iters}")
    print(f"Average Time: {avg_time_ms:.4f} ms")
    print(f"Std Dev     : {std_dev_ms:.4f} ms")
    print(f"--------------------------------------------------")

if __name__ == "__main__":
    benchmark_e8p_cache_wh_with_stats()

Project root added to path: /home/jgryu/workspace/weight_compression/quip-sharp
Initializing Benchmark for 4096x4096 Tensor...
Quantizing tensor...
Fast quantizing for benchmark (skipping Hessian/LDLQ)...
Determining K and Hadamard matrices using get_hadK...
K_left: 1, K_right: 1
Benchmarking cache_WH on cuda:0...
--------------------------------------------------
Task: cache_WH Execution
Matrix Size: 4096x4096
K Parameters: Left=1, Right=1
Iterations: 100
Average Time: 3.0191 ms
Std Dev     : 2.4619 ms
--------------------------------------------------


In [18]:
import time
import torch
import torch.nn as nn
import sys
import os

project_root = "/home/jgryu/workspace/weight_compression/quip-sharp"
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root added to path: {project_root}")


from latticee8_padded12_rvq4bit import E8P12RVQ4B_codebook, QuantizedE8P12RVQ4BLinear, matmul_hadU_cuda
# (편의를 위해 위 클래스들이 이미 정의되어 있다고 가정하고 진행합니다.)
from lib.utils import get_hadK


def fast_quantize_for_bench(layer, codebook, W):
    """
    디코딩 속도 측정을 위해 복잡한 QUIP 알고리즘(Hessian, LDLQ 등)을 건너뛰고
    단순히 가중치를 양자화하여 Qidxs_list를 생성합니다.
    """
    print("Fast quantizing for benchmark (skipping Hessian/LDLQ)...")
    
    # 1. 가중치 형태 확인 (Out_features, In_features)
    # E8P 코드북은 마지막 차원이 8(codesz)이어야 작동하므로 view로 형태를 맞춰줍니다.
    # W가 (4096, 4096)이라면 -> (4096 * 4096 // 8, 8)로 변환하여 처리
    orig_shape = W.shape
    flattened_W = W.view(-1, codebook.codesz)
    
    # 2. 단순 양자화 수행 (cb.quantize 직접 호출)
    # return_idx=True로 설정하여 값과 인덱스를 모두 받습니다.
    # resid_scale_override는 기본값(-1)을 사용하거나 필요시 지정합니다.
    with torch.no_grad():
        _, idxs = codebook.quantize(flattened_W, return_idx=True)
    
    # 3. 인덱스 형태 복원
    # (Total_Blocks, ) 형태를 (Out, In // codesz) 형태로 다시 맞춤
    # E8P의 경우 idxs는 (N, ) 형태일 수 있으므로 차원 확인 필요
    # 일반적으로 codebook.quantize는 입력의 앞쪽 차원을 유지합니다.
    # 여기서는 안전하게 원래 레이어가 기대하는 shape으로 맞춥니다.
    
    # idxs의 현재 shape 확인
    # E8P12RVQ4B_codebook의 quantize는 입력 shape (..., 8) -> 출력 idxs (..., ) (마지막 차원 축소)
    # 따라서 (4096, 4096) -> (4096, 512) 형태의 인덱스가 나옵니다 (4096/8 = 512)
    idxs = idxs.view(orig_shape[0], orig_shape[1] // codebook.codesz)

    # 4. 인덱스 패킹 (Packing)
    # 16비트 인덱스(2개의 12비트 코드)를 32비트 등으로 패킹
    packed_idxs = codebook.maybe_pack_idxs(idxs)
    
    # 5. 레이어용 언패킹 (Unpacking for Layer)
    # Residual 부분과 Main 부분을 분리하여 리스트로 반환
    Qidxs_list = layer.maybe_unpack_idxs(packed_idxs)
    
    return Qidxs_list

def benchmark_e8p_cache_wh_with_k():
    # 1. 설정 (Setup)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if device.type != 'cuda':
        print("Warning: CUDA is not available. Benchmark results will be invalid.")
        return

    n = 4096
    m = 4096
    
    print(f"Initializing Benchmark for {n}x{m} Tensor...")

    # 2. 모델 및 코드북 초기화
    # 학습용 코드북(양자화 수행용)과 추론용 레이어(벤치마크 대상) 생성
    quantizer_codebook = E8P12RVQ4B_codebook(inference=False).to(device)
    linear_layer = QuantizedE8P12RVQ4BLinear(device).to(device)

    # 3. 더미 데이터 생성 및 양자화
    print("Quantizing tensor...")
    W = torch.randn(n, m, device=device, dtype=torch.float32)
    
    Qidxs_list = fast_quantize_for_bench(linear_layer, quantizer_codebook, W)

    # 4. QuantizedLinear 로직을 참고하여 K 및 Hadamard 행렬 결정
    print("Determining K and Hadamard matrices using get_hadK...")
    
    # QuantizedLinear.__init__ 로직 재현:
    # had_left_T, K_left = get_hadK(in_features)
    # had_right, K_right = get_hadK(out_features)
    
    had_left_raw, K_left = get_hadK(n)
    had_right_raw, K_right = get_hadK(m)

    # QuantizedLinear.no_ckpt_forward 에서 cache_WH를 호출할 때:
    # self.codebook_class.cache_WH(..., self.had_left_T.T.contiguous(), self.had_right, ...)
    # __init__에서 had_left가 transpose되어 저장되므로, 호출 시 다시 transpose하여 원본 형태로 복구됨.
    # 따라서 여기서는 get_hadK의 출력을 그대로(contiguous하게) 사용하면 됩니다.
    
    had_left = had_left_raw.to(device).contiguous() if had_left_raw is not None else None
    had_right = had_right_raw.to(device).contiguous() if had_right_raw is not None else None
    
    print(f"K_left: {K_left}, K_right: {K_right}")

    # 5. 벤치마크 (Benchmark cache_WH)
    print(f"Benchmarking cache_WH on {device}...")
    # Warmup
    for _ in range(100):
        linear_layer.cache_WH(
            n=n, m=m, 
            Qidxs_list=Qidxs_list, 
            had_left=had_left, 
            had_right=had_right, 
            K_left=K_left, 
            K_right=K_right
        )
    torch.cuda.synchronize()

    # 측정 루프
    num_iters = 100
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    for _ in range(num_iters):
        linear_layer.cache_WH(
            n=n, m=m, 
            Qidxs_list=Qidxs_list, 
            had_left=had_left, 
            had_right=had_right, 
            K_left=K_left, 
            K_right=K_right
        )
    end_event.record()
    torch.cuda.synchronize()

    # 결과 계산
    elapsed_time_ms = start_event.elapsed_time(end_event)
    avg_time_ms = elapsed_time_ms / num_iters

    print(f"--------------------------------------------------")
    print(f"Matrix Size: {n}x{m}")
    print(f"K Parameters: Left={K_left}, Right={K_right}")
    print(f"Average cache_WH execution time: {avg_time_ms:.4f} ms")
    print(f"--------------------------------------------------")

if __name__ == "__main__":
    benchmark_e8p_cache_wh_with_k()

Project root added to path: /home/jgryu/workspace/weight_compression/quip-sharp
Initializing Benchmark for 4096x4096 Tensor...
Quantizing tensor...
Fast quantizing for benchmark (skipping Hessian/LDLQ)...
Determining K and Hadamard matrices using get_hadK...
K_left: 1, K_right: 1
Benchmarking cache_WH on cuda:0...
--------------------------------------------------
Matrix Size: 4096x4096
K Parameters: Left=1, Right=1
Average cache_WH execution time: 4.5497 ms
--------------------------------------------------


In [17]:
import time
import torch
import torch.nn as nn
import sys
import os

# 1. 프로젝트 루트 경로 설정 (기존과 동일)
project_root = "/home/jgryu/workspace/weight_compression/quip-sharp"
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. 필요한 모듈 임포트
from latticee8_padded12_rvq4bit import E8P12RVQ4B_codebook, QuantizedE8P12RVQ4BLinear
from lib.utils import get_hadK

def fast_quantize_for_bench(layer, codebook, W):
    """
    벤치마크용 더미 Qidxs 생성 함수
    """
    print("Fast quantizing for benchmark (skipping Hessian/LDLQ)...")
    orig_shape = W.shape
    flattened_W = W.view(-1, codebook.codesz)
    
    with torch.no_grad():
        _, idxs = codebook.quantize(flattened_W, return_idx=True)
    
    idxs = idxs.view(orig_shape[0], orig_shape[1] // codebook.codesz)
    packed_idxs = codebook.maybe_pack_idxs(idxs)
    Qidxs_list = layer.maybe_unpack_idxs(packed_idxs)
    
    return Qidxs_list

def benchmark_forward_latency():
    # -------------------------------------------------------
    # 1. 설정 (Setup)
    # -------------------------------------------------------
    device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
    n = 4096  # Input Features
    m = 4096  # Output Features
    
    print(f"Initializing Forward Benchmark for {n}x{m} Layer...")
    print(f"Device: {device}")

    # -------------------------------------------------------
    # 2. 모델 및 코드북 준비
    # -------------------------------------------------------
    quantizer_codebook = E8P12RVQ4B_codebook(inference=False).to(device)
    # 실제 추론용 레이어 (inference=True 내부적으로 처리됨)
    linear_layer = QuantizedE8P12RVQ4BLinear(device).to(device)

    # -------------------------------------------------------
    # 3. 데이터 준비 (Weights & Input)
    # -------------------------------------------------------
    # 3-1. 가중치 양자화 (더미 데이터)
    print("Preparing dummy weights...")
    W = torch.randn(n, m, device=device, dtype=torch.float32)
    Qidxs_list = fast_quantize_for_bench(linear_layer, quantizer_codebook, W)

    # 3-2. Hadamard 행렬 및 K 파라미터 준비
    # QuantizedLinear의 forward는 had_left_T(Transposed)를 기대하는 경우가 많으므로
    # 기존 코드의 관례에 따라 변환해줍니다.
    had_left_raw, K_left = get_hadK(n)
    had_right_raw, K_right = get_hadK(m)

    had_left_T = had_left_raw.to(device).T.contiguous() if had_left_raw is not None else None
    had_right = had_right_raw.to(device).contiguous() if had_right_raw is not None else None

    # 3-3. SU, SV 벡터 (Incoherence용 Scaling Vector)
    # 실제로는 최적화된 값이지만, 속도 측정엔 1이나 랜덤이면 충분합니다.
    SU = torch.ones(n, device=device, dtype=torch.float16)
    SV = torch.ones(m, device=device, dtype=torch.float16)

    # 3-4. 입력 데이터 (가장 간단한 형태: Batch=1, Seq=1)
    # LLM의 Token Generation 단계 시나리오
    batch_size = 1
    seq_len = 1
    input_tensor = torch.randn(batch_size, seq_len, n, device=device, dtype=torch.float32)

    print(f"Input Shape: {input_tensor.shape}")

    # -------------------------------------------------------
    # 4. 벤치마크 (Benchmark Forward Pass)
    # -------------------------------------------------------
    print(f"Benchmarking Forward Pass...")

    # Warmup
    for _ in range(10):
        with torch.no_grad():
            output = linear_layer.forward(
                input=input_tensor,
                Qidxs_list=Qidxs_list,
                SU=SU,
                SV=SV,
                had_left_T=had_left_T,
                had_right=had_right,
                K_left=K_left,
                K_right=K_right,
                rescale_WH=False # 벤치마크용 단순화
            )
    torch.cuda.synchronize()

    # 측정
    num_iters = 100
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    with torch.no_grad():
        for _ in range(num_iters):
            output = linear_layer.forward(
                input=input_tensor,
                Qidxs_list=Qidxs_list,
                SU=SU,
                SV=SV,
                had_left_T=had_left_T,
                had_right=had_right,
                K_left=K_left,
                K_right=K_right,
                rescale_WH=False
            )
    end_event.record()
    torch.cuda.synchronize()

    # -------------------------------------------------------
    # 5. 결과 출력
    # -------------------------------------------------------
    elapsed_time_ms = start_event.elapsed_time(end_event)
    avg_time_ms = elapsed_time_ms / num_iters

    print(f"--------------------------------------------------")
    print(f"Layer Size: {n} -> {m}")
    print(f"Input Size: {input_tensor.shape} (Batch=1 latency)")
    print(f"Average Forward execution time: {avg_time_ms:.4f} ms")
    print(f"Output Shape: {output.shape}")
    print(f"--------------------------------------------------")

if __name__ == "__main__":
    benchmark_forward_latency()

Initializing Forward Benchmark for 4096x4096 Layer...
Device: cuda:3
Preparing dummy weights...
Fast quantizing for benchmark (skipping Hessian/LDLQ)...
Input Shape: torch.Size([1, 1, 4096])
Benchmarking Forward Pass...
--------------------------------------------------
Layer Size: 4096 -> 4096
Input Size: torch.Size([1, 1, 4096]) (Batch=1 latency)
Average Forward execution time: 9.1683 ms
Output Shape: torch.Size([1, 1, 4096])
--------------------------------------------------
